In [2]:
!pip install google-api-python-client
!pip install youtube-transcript-api openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 9.4 MB/s eta 0:00:00


In [ ]:
from googleapiclient.discovery import build
import pandas as pd

# 🔑 Replace with your API key
API_KEY = "your_key"

youtube = build("youtube", "v3", developerKey=API_KEY)

QUERY = "Full Python Course"
MAX_VIDEOS = 20
MAX_COMMENTS = 100

videos_data = []
comments_data = []

# 🔍 Search videos
search_response = youtube.search().list(
    q=QUERY,
    part="id,snippet",
    type="video",
    maxResults=MAX_VIDEOS
).execute()

for item in search_response["items"]:
    video_id = item["id"]["videoId"]
    title = item["snippet"]["title"]
    channel = item["snippet"]["channelTitle"]

    # 📊 Video metadata
    video_response = youtube.videos().list(
        part="statistics,contentDetails,snippet",
        id=video_id
    ).execute()

    video_info = video_response["items"][0]
    stats = video_info["statistics"]
    details = video_info["contentDetails"]
    snippet = video_info["snippet"]

    videos_data.append({
    "video_id": video_id,
    "video_title": title,
    "description": snippet.get("description", ""),
    "channel_name": channel,
    "views": int(stats.get("viewCount", 0)),
    "likes": int(stats.get("likeCount", 0)),
    "duration": details["duration"],
    "publish_date": snippet["publishedAt"],
    "video_url": f"https://www.youtube.com/watch?v={video_id}"
})

    # 💬 Comments
    try:
        comment_response = youtube.commentThreads().list(
            part="snippet",
            videoId=video_id,
            maxResults=MAX_COMMENTS,
            textFormat="plainText"
        ).execute()

        for c in comment_response.get("items", []):
            comment = c["snippet"]["topLevelComment"]["snippet"]
            comments_data.append({
                "video_id": video_id,
                "comment_id": c["id"],
                "comment_text": comment["textDisplay"],
                "timestamp": comment["publishedAt"]
            })

    except:
        print(f"Comments disabled for video {video_id}")

# 💾 Save data
videos_df = pd.DataFrame(videos_data)
comments_df = pd.DataFrame(comments_data)

videos_df.to_csv("youtube_videos.csv", index=False)
comments_df.to_csv("youtube_comments.csv", index=False)

print("Phase 1 completed")
print("Videos collected:", len(videos_df))
print("Comments collected:", len(comments_df))


Comments disabled for video YvYKcGAfJvs
Phase 1 completed
Videos collected: 25
Comments collected: 2033


In [4]:
import pandas as pd
import re

# Load raw comments
comments_df = pd.read_csv("youtube_comments.csv")

# Text cleaning function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)           # remove URLs
    text = re.sub(r"[^a-z0-9\s]", " ", text)      # remove special chars
    text = re.sub(r"\s+", " ", text).strip()      # normalize spaces
    return text

comments_df["clean_comment"] = comments_df["comment_text"].astype(str).apply(clean_text)
comments_df["comment_length"] = comments_df["clean_comment"].apply(lambda x: len(x.split()))

# Save cleaned comments
comments_df.to_csv("youtube_comments_cleaned.csv", index=False)

print("Phase 2 completed")
print("Total cleaned comments:", len(comments_df))
comments_df.head()


Phase 2 completed
Total cleaned comments: 2033


,video_id,comment_id,comment_text,timestamp,clean_comment,comment_length
0,K5KVEU3aaeQ,Ugyz70nN7eD2hYKS4Rt4AaABAg,🔥Want to master Python? Get my Python mastery ...,2025-02-12T18:50:18Z,want to master python get my python mastery co...,26
1,K5KVEU3aaeQ,Ugyg2261psx-1Ap26Yx4AaABAg,I can't even get helloworld to print. i downlo...,2026-03-10T22:13:34Z,i can t even get helloworld to print i downloa...,25
2,K5KVEU3aaeQ,UgyfuVjTxh1dHyMGziV4AaABAg,"Tämä tuli tarpeeseen, kiitos.",2026-03-10T18:49:42Z,t m tuli tarpeeseen kiitos,5
3,K5KVEU3aaeQ,Ugz5q_PmRseBto3d78h4AaABAg,You are really good!,2026-03-09T18:25:40Z,you are really good,4
4,K5KVEU3aaeQ,Ugz2zuflhaxbosuPrtZ4AaABAg,ur amazing man,2026-03-09T15:25:41Z,ur amazing man,3


In [5]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer


In [ ]:
comments_df = pd.read_csv("/content/youtube_comments_cleaned.csv")

# Sanity check
assert "clean_comment" in comments_df.columns
semantic_query = "Complete Python programming tutorial explaining concepts and problem solving"
model = SentenceTransformer("intfloat/e5-base-v2")
comments = comments_df["clean_comment"].fillna("").tolist()

comment_embeddings = model.encode(
    ["passage: " + c for c in comments],
    normalize_embeddings=True,
    batch_size=32,
    show_progress_bar=True
)
query_embedding = model.encode(
    ["query: " + semantic_query],
    normalize_embeddings=True
)
relevance_scores = np.dot(
    comment_embeddings,
    query_embedding.T
).squeeze()


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/64 [00:00<?, ?it/s]

In [ ]:
comments_df["relevance_score"] = relevance_scores
comments_df = comments_df.sort_values(
    by="relevance_score",
    ascending=False
)
comments_df.to_csv(
    "phase3_comments_relevance.csv",
    index=False
)


In [ ]:
phase3_df = pd.read_csv("phase3_comments_relevance.csv")
TOP_K = 1000  # justified as top-K semantic filtering

relevant_df = (
    phase3_df
    .sort_values("relevance_score", ascending=False)
    .head(TOP_K)
    .copy()
)

print("Selected relevant comments:", len(relevant_df))


Selected relevant comments: 1000


In [ ]:
from transformers import pipeline

sentiment_model = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",
    truncation=True,
    max_length=512
)

print("Sentiment model loaded successfully")


config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/501M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


model.safetensors:   0%|          | 0.00/501M [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Sentiment model loaded successfully


In [ ]:
sentiment_model("This course is absolutely amazing!")


[{'label': 'positive', 'score': 0.9776747822761536}]

In [ ]:
def sentiment_score(text):
    result = sentiment_model(text)[0]

    label = result["label"].lower()
    score = result["score"]

    if label == "positive":
        return score
    elif label == "negative":
        return -score
    else:
        return 0.0


In [ ]:
print(sentiment_score("This course is amazing"))
print(sentiment_score("This tutorial is terrible"))
print(sentiment_score("This is a Python video"))


0.9751406908035278
-0.9413253664970398
0.0


In [ ]:
# Ensure clean_comment is string and not null
relevant_df = relevant_df.dropna(subset=["clean_comment"])
relevant_df["clean_comment"] = relevant_df["clean_comment"].astype(str)

print("Remaining relevant comments:", len(relevant_df))


Remaining relevant comments: 968


In [ ]:
def safe_sentiment_score(text):
    try:
        return sentiment_score(text)
    except:
        return 0.0   # neutral fallback


In [ ]:
relevant_df["sentiment_score"] = relevant_df["clean_comment"].apply(safe_sentiment_score)

relevant_df[["clean_comment", "relevance_score", "sentiment_score"]].head()


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


,clean_comment,relevance_score,sentiment_score
0,python course,0.866156,0.000000
1,the amazing tutorial for beginners in python t...,0.853168,0.981482
2,does this course cover whole python concept,0.850329,0.000000
3,half of this is just python basics,0.847672,-0.629851
4,the tutorial is brilliant as usual my friend s...,0.845419,0.982108


In [ ]:
relevant_df["sentiment_score"].describe()


,sentiment_score
count,968.000000
mean,0.226220
std,0.524840
min,-0.945177
25%,0.000000
50%,0.000000
75%,0.840538
max,0.990134


In [ ]:
relevant_df = relevant_df.copy()
relevant_df.loc[:, "sentiment_score"] = relevant_df["clean_comment"].apply(safe_sentiment_score)


In [ ]:
# Finalize Phase 4 output
relevant_df.to_csv(
    "phase4_relevant_comments_sentiment.csv",
    index=False
)

print("✅ Phase 4 finalized and saved successfully")
print("Rows saved:", len(relevant_df))


✅ Phase 4 finalized and saved successfully
Rows saved: 968


PHASE 5 – Emotion-Weighted Engagement Formulation (CORE NOVELTY)

In [ ]:
import numpy as np
import pandas as pd
relevant_df = pd.read_csv("/content/phase4_relevant_comments_sentiment.csv")

relevant_df.head()
relevant_df["emotion_weighted_score"] = (
    relevant_df["relevance_score"] * relevant_df["sentiment_score"]
)


In [ ]:
relevant_df["emotion_weighted_score"] = (
    relevant_df["relevance_score"] * relevant_df["sentiment_score"]
)


In [ ]:
video_emotion_df = (
    relevant_df
    .groupby("video_id")
    .agg(
        emotional_engagement=("emotion_weighted_score", "mean"),
        emotional_variance=("emotion_weighted_score", "var"),
        relevant_comment_count=("emotion_weighted_score", "count")
    )
    .reset_index()
)


In [ ]:
video_emotion_df.head()


,video_id,emotional_engagement,emotional_variance,relevant_comment_count
0,8DvywoWv6fI,0.159138,0.193765,43
1,AQJbHAeUhzM,0.002459,0.134270,47
2,H2EJuAcrZYU,0.175862,0.242123,56
3,K5KVEU3aaeQ,0.158943,0.203170,48
4,Onjs26YvfIQ,0.000000,0.000000,4


In [ ]:
video_emotion_df.to_csv("phase5_video_emotional_engagement.csv", index=False)
print("✅ Phase 5 completed successfully")


✅ Phase 5 completed successfully


PHASE 6: FINAL ENGAGEMENT SCORE & VIDEO RANKING

In [ ]:
import pandas as pd
from sklearn.preprocessing import MinMaxScaler

# Load Phase 5 data
phase5 = pd.read_csv("/content/phase5_video_emotional_engagement.csv")

# Load video metadata
videos = pd.read_csv("youtube_videos.csv")

# Merge on video_id
df = phase5.merge(videos, on="video_id", how="inner")

# Select columns for normalization
scaler = MinMaxScaler()

df[["likes_norm", "comment_count_norm"]] = scaler.fit_transform(
    df[["likes", "relevant_comment_count"]]
)

# Compute Final Engagement Score
df["final_engagement_score"] = (
    0.4 * df["emotional_engagement"] +
    0.2 * (1 - df["emotional_variance"]) +
    0.2 * df["likes_norm"] +
    0.2 * df["comment_count_norm"]
)

# Rank videos
df["rank"] = df["final_engagement_score"].rank(
    ascending=False, method="dense"
)

# Sort by rank
df = df.sort_values("rank")

# Save Phase 6 output
df[["video_id", "final_engagement_score", "rank"]].to_csv(
    "phase6_final_video_ranking.csv", index=False
)

df.head()
df.to_csv("phase6_full.csv", index=False)


PHASE 7 – Public Transcript Only Pipeline

single new phase 7

In [ ]:
# ==========================================================
# PHASE 7 (FINAL UPDATED VERSION)
# Intrinsic Teaching Quality Score (ITQS)
# ==========================================================
!pip install isodate
import pandas as pd
import numpy as np
import re
import isodate
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import MinMaxScaler
import google.generativeai as genai
import json

# -------------------------------
# 1️⃣ Load Data
# -------------------------------
videos_df = pd.read_csv("youtube_videos.csv")
print("Total videos:", len(videos_df))

# -------------------------------
# 2️⃣ Combine Title + Description
# -------------------------------
videos_df["content_text"] = (
    videos_df["video_title"].fillna("") + " " +
    videos_df["description"].fillna("")
)

# -------------------------------
# 3️⃣ Semantic Alignment (E5)
# -------------------------------
model_embed = SentenceTransformer("intfloat/e5-base-v2")

semantic_query = "Complete Python programming tutorial explaining concepts and problem solving"

query_embedding = model_embed.encode(
    ["query: " + semantic_query],
    normalize_embeddings=True
)

content_embeddings = model_embed.encode(
    ["passage: " + t for t in videos_df["content_text"]],
    normalize_embeddings=True
)

videos_df["semantic_alignment"] = np.dot(
    content_embeddings,
    query_embedding.T
).squeeze()

# -------------------------------
# 4️⃣ Depth Score (Duration-based)
# -------------------------------
videos_df["duration_minutes"] = videos_df["duration"].apply(
    lambda x: isodate.parse_duration(x).total_seconds() / 60
)

scaler = MinMaxScaler()
videos_df["depth_score"] = scaler.fit_transform(
    videos_df[["duration_minutes"]]
)

# -------------------------------
# 5️⃣ Lexical Diversity
# -------------------------------
def lexical_diversity(text):
    words = re.findall(r'\w+', text.lower())
    if len(words) == 0:
        return 0
    return len(set(words)) / len(words)

videos_df["lexical_diversity"] = videos_df["content_text"].apply(lexical_diversity)
videos_df["lexical_diversity"] = scaler.fit_transform(
    videos_df[["lexical_diversity"]]
)

# -------------------------------
# 6️⃣ Batch LLM Teaching Score (STRICT JSON VERSION)
# -------------------------------

genai.configure(api_key="your_key")
model_llm = genai.GenerativeModel("models/gemini-2.5-flash")

video_blocks = ""

for i, row in videos_df.iterrows():
    video_blocks += f"""
Video {i}
Title: {row['video_title']}
Description: {row['description']}
"""

prompt = f"""
Return ONLY valid JSON.
Do NOT include explanations.
Do NOT include markdown.
Do NOT include text before or after JSON.

Format:
{{ "0": number, "1": number, ... }}

Each number must be between 0 and 10.

Evaluate educational teaching quality using:
- Clear learning objective
- Structured explanation
- Logical concept progression
- Example usage
- Beginner friendliness
- Practical problem solving

Videos:
{video_blocks}
"""

response = model_llm.generate_content(prompt)

raw_text = response.text.strip()

# Extract JSON block safely
import re

json_match = re.search(r'\{.*\}', raw_text, re.DOTALL)

if json_match:
    json_text = json_match.group(0)
    scores = json.loads(json_text)
else:
    print("⚠ JSON block not found. Using fallback.")
    scores = {str(i): 5 for i in videos_df.index}

videos_df["teaching_score_raw"] = videos_df.index.map(
    lambda x: scores.get(str(x), 5)
)

videos_df["teaching_score"] = videos_df["teaching_score_raw"] / 10
# -------------------------------
# 7️⃣ Final Intrinsic Teaching Quality Score
# -------------------------------
videos_df["ITQS"] = (
    0.30 * videos_df["semantic_alignment"] +
    0.20 * videos_df["depth_score"] +
    0.20 * videos_df["lexical_diversity"] +
    0.30 * videos_df["teaching_score"]
)

# -------------------------------
# 8️⃣ Save Output
# -------------------------------
videos_df.to_csv("phase7_intrinsic_teaching_quality.csv", index=False)

print("✅ Phase 7 Updated Successfully")
print(videos_df[["video_id", "ITQS"]].head())

PHASE 8 — Adaptive Hybrid Ranking

new pahse 8


In [ ]:
# ==========================================================
# PHASE 8 – BALANCED HYBRID RANKING (FINAL VERSION)
# ==========================================================

import pandas as pd
import numpy as np

# -------------------------------
# 1️⃣ Load Required Phases
# -------------------------------
phase5 = pd.read_csv("phase5_video_emotional_engagement.csv")
phase6 = pd.read_csv("phase6_final_video_ranking.csv")
phase7 = pd.read_csv("phase7_intrinsic_teaching_quality.csv")  # NEW ITQS

# -------------------------------
# 2️⃣ Merge Data
# -------------------------------
df = phase6.merge(
    phase5[["video_id", "relevant_comment_count"]],
    on="video_id",
    how="inner"
).merge(
    phase7[["video_id", "ITQS"]],
    on="video_id",
    how="inner"
)

print("Merged videos:", len(df))

# -------------------------------
# 3️⃣ Adaptive Alpha (Non-Linear)
# -------------------------------
# Engagement strength
df["engagement_strength"] = np.log1p(df["relevant_comment_count"])

# Balance tuning parameter
k = 5   # try values between 2–5 for tuning

df["alpha"] = df["engagement_strength"] / (
    df["engagement_strength"] + k
)

# -------------------------------
# 4️⃣ Hybrid Score
# -------------------------------
df["hybrid_score"] = (
    df["alpha"] * df["final_engagement_score"] +
    (1 - df["alpha"]) * df["ITQS"]
)

# -------------------------------
# 5️⃣ Final Ranking
# -------------------------------
df["final_rank"] = df["hybrid_score"].rank(
    ascending=False,
    method="dense"
)

df = df.sort_values("final_rank")

# -------------------------------
# 6️⃣ Save Output
# -------------------------------
df.to_csv("phase8_hybrid_final_ranking.csv", index=False)

print("✅ Phase 8 Completed Successfully")
print(df[["video_id", "hybrid_score", "final_rank"]].head())

In [ ]:
# Contribution analysis
df["es_contribution"] = df["alpha"] * df["final_engagement_score"]
df["itqs_contribution"] = (1 - df["alpha"]) * df["ITQS"]

print("ES dominant:", (df["es_contribution"] > df["itqs_contribution"]).sum())
print("ITQS dominant:", (df["itqs_contribution"] > df["es_contribution"]).sum())

print("\nAlpha distribution:")
print(df["alpha"].describe())

In [ ]:
from scipy.stats import spearmanr

hybrid_rank = df["hybrid_score"].rank(ascending=False)
es_rank = df["final_engagement_score"].rank(ascending=False)
itqs_rank = df["ITQS"].rank(ascending=False)

print("Hybrid vs ES:", spearmanr(hybrid_rank, es_rank))
print("Hybrid vs ITQS:", spearmanr(hybrid_rank, itqs_rank))

LLm

In [ ]:
import google.generativeai as genai
import os
import pandas as pd

# Set Google API key
genai.configure(api_key="your_key")

# Load model
model = genai.GenerativeModel("models/gemini-2.5-flash")

phase5 = pd.read_csv("/content/phase5_video_emotional_engagement.csv")
phase6 = pd.read_csv("/content/phase6_final_video_ranking.csv")
phase7 = pd.read_csv("/content/phase7_intrinsic_teaching_quality.csv")
phase8 = pd.read_csv("/content/phase8_hybrid_final_ranking.csv")

# Merge safely
df = phase8.merge(phase6, on="video_id", how="inner") \
           .merge(phase7, on="video_id", how="inner") \
           .merge(phase5, on="video_id", how="inner")

print(df.columns)
df.head()

In [ ]:
print(df.columns)

In [ ]:
df_clean = df.copy()

df_clean["final_engagement_score"] = df_clean["final_engagement_score_x"]
df_clean["ITQS"] = df_clean["ITQS_x"]
df_clean["relevant_comment_count"] = df_clean["relevant_comment_count_x"]

df_clean = df_clean[
    [
        "video_id",
        "hybrid_score",
        "final_rank",
        "alpha",  # ← ADD THIS
        "final_engagement_score",
        "ITQS",
        "relevant_comment_count"
    ]
]

df_clean.head()

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import time

def generate_explanation(row, retries=3):

    # Compute actual contributions
    es_contribution = row["alpha"] * row["final_engagement_score"]
    itqs_contribution = (1 - row["alpha"]) * row["ITQS"]

    dominant = "Engagement Score (ES)" if es_contribution > itqs_contribution else "Intrinsic Teaching Quality Score (ITQS)"

    prompt = f"""
    A hybrid ranking system computed the following:

    Hybrid Score = alpha * ES + (1 - alpha) * ITQS

    alpha: {row['alpha']:.3f}
    ES: {row['final_engagement_score']:.3f}
    ITQS: {row['ITQS']:.3f}

    ES contribution: {es_contribution:.3f}
    ITQS contribution: {itqs_contribution:.3f}

    Relevant Comment Count: {row['relevant_comment_count']}

    The dominant factor is: {dominant}

    Explain analytically why this video received its rank based ONLY on these values.
    Do not invent additional reasons.
    Keep explanation technical and concise.
    """

    for attempt in range(retries):
        try:
            response = model.generate_content(prompt)
            return response.text
        except Exception as e:
            print(f"Retry {attempt+1} due to error: {e}")
            time.sleep(2)

    return "Explanation generation failed."

In [ ]:
top5 = df_clean.sort_values("final_rank").head(5).copy()

top5["explanation"] = top5.apply(generate_explanation, axis=1)

top5[["video_id", "final_rank", "explanation"]]

validation

PHASE 1 — Data Collection Validation

Coverage

In [ ]:
print("Unique videos:", videos_df["video_id"].nunique())
print("Unique comments:", comments_df["comment_id"].nunique())

Missing Data Check

In [ ]:
print(videos_df.isnull().sum())
print(comments_df.isnull().sum())

Comment Distribution Plot

In [ ]:
import matplotlib.pyplot as plt

comments_df.groupby("video_id").size().plot(kind="bar")
plt.title("Comments per Video")
plt.show()

PHASE 2 — Cleaning Validation

Before vs After Length

In [ ]:
comments_df["original_len"] = comments_df["comment_text"].astype(str).apply(len)
comments_df["clean_comment"] = comments_df["clean_comment"].fillna("").astype(str)

comments_df["original_len"] = comments_df["comment_text"].fillna("").astype(str).apply(len)
comments_df["clean_len"] = comments_df["clean_comment"].apply(len)

print(comments_df[["original_len","clean_len"]].describe())

#print(comments_df[["original_len","clean_len"]].describe())

In [ ]:
print("Empty cleaned comments:",
      (comments_df["clean_comment"].str.strip() == "").sum())

In [ ]:
import matplotlib.pyplot as plt

plt.hist(comments_df["clean_len"], bins=30)
plt.title("Cleaned Comment Length Distribution")
plt.xlabel("Length")
plt.ylabel("Frequency")
plt.show()

PHASE 3 — Semantic Relevance Validation

Check Range

In [ ]:
print(comments_df["relevance_score"].describe())

In [ ]:
plt.hist(comments_df["relevance_score"], bins=30)
plt.title("Relevance Score Distribution")
plt.show()

PHASE 4 — Sentiment Validation

In [ ]:
print(relevant_df["sentiment_score"].describe())

In [ ]:
plt.hist(relevant_df["sentiment_score"], bins=30)
plt.title("Sentiment Score Distribution")
plt.show()

PHASE 5 — Emotional Engagement Validation

Check Bounds

In [ ]:
print(relevant_df["emotion_weighted_score"].describe())

In [ ]:
plt.hist(relevant_df["emotion_weighted_score"], bins=30)
plt.title("emotion_weighted_score Distribution")
plt.show()

emotional_variance

In [ ]:
print(video_emotion_df["emotional_variance"].describe())

PHASE 6 — Engagement Score Validation

check range

In [ ]:
print("Available columns:", df.columns)


In [ ]:
print(df["final_engagement_score_x"].describe())

Correlation with Likes

In [ ]:
from scipy.stats import spearmanr
phase6_full = pd.read_csv("phase6_full.csv")
from scipy.stats import spearmanr

print(spearmanr(
    phase6_full["likes"],
    phase6_full["final_engagement_score"]
))

Correlation with comment count

In [ ]:
print(spearmanr(
    phase6_full["relevant_comment_count"],
    phase6_full["final_engagement_score"]
))

Correlation with emotional engagement

In [ ]:
print(spearmanr(
    phase6_full["emotional_engagement"],
    phase6_full["final_engagement_score"]
))

In [ ]:
from scipy.stats import spearmanr

metrics = [
    "likes",
    "relevant_comment_count",
    "emotional_engagement",
    "emotional_variance"
]

for m in metrics:
    corr = spearmanr(phase6_full[m], phase6_full["final_engagement_score"])
    print(f"{m} vs ES:", corr)

PHASE 7 — ITQS Validation

In [ ]:
print(videos_df["ITQS"].describe())

In [ ]:
print(spearmanr(videos_df["semantic_alignment"], videos_df["ITQS"]))
print(spearmanr(videos_df["teaching_score"], videos_df["ITQS"]))

PHASE 8 — Hybrid Validation

check bounds

In [ ]:
print(df["hybrid_score"].describe())

In [ ]:
import seaborn as sns

corr = df[["final_engagement_score_x",
           "ITQS_x",
           "hybrid_score"]].corr(method="spearman")
sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Spearman Correlation Matrix")
plt.show()

Top-K Overlap Analysis

1️⃣ Rank Stability (Bootstrap)

Resample comments → recompute ES → recompute hybrid → measure rank change.

That proves robustness.

2️⃣ Sensitivity Analysis of k

Plot α as function of comment count.

In [ ]:
top_es = set(df.sort_values("final_engagement_score_x", ascending=False).head(5)["video_id"])
top_itqs = set(df.sort_values("ITQS_x", ascending=False).head(5)["video_id"])
top_hybrid = set(df.sort_values("hybrid_score", ascending=False).head(5)["video_id"])

print("ES-Hybrid overlap:", len(top_es & top_hybrid))
print("ITQS-Hybrid overlap:", len(top_itqs & top_hybrid))

testing

In [ ]:
df_analysis = df_clean.copy()

df_analysis["rank_ES"] = df_analysis["final_engagement_score"].rank(ascending=False, method="dense")
df_analysis["rank_ITQS"] = df_analysis["ITQS"].rank(ascending=False, method="dense")
df_analysis["rank_Hybrid"] = df_analysis["hybrid_score"].rank(ascending=False, method="dense")

df_analysis.head()

In [ ]:
high_itqs = df_analysis.sort_values("rank_ITQS").head(5)
high_itqs

In [ ]:
df_analysis["hybrid_vs_ES_shift"] = df_analysis["rank_ES"] - df_analysis["rank_Hybrid"]
df_analysis["hybrid_vs_ITQS_shift"] = df_analysis["rank_ITQS"] - df_analysis["rank_Hybrid"]

df_analysis.sort_values("hybrid_vs_ES_shift", ascending=False).head()

In [ ]:
import matplotlib.pyplot as plt

plt.scatter(df_analysis["rank_ES"], df_analysis["rank_Hybrid"])
plt.xlabel("Engagement Rank")
plt.ylabel("Hybrid Rank")
plt.title("Hybrid vs Engagement Rank")
plt.show()

In [ ]:
improved = (df_analysis["hybrid_vs_ES_shift"] > 0).sum()
print("Videos improved over ES ranking:", improved)

In [ ]:
df_analysis["hybrid_vs_ES_shift"].mean()

BASE LINE COMPARISION

Likes-Based Ranking (Baseline 1)

In [ ]:
df["rank_likes_baseline"] = df["likes"].rank(
    ascending=False,
    method="dense"
)

df[["video_id","video_title","likes","rank_likes_baseline"]].sort_values("rank_likes_baseline").head()

Comment Count Ranking (Baseline 2)

In [ ]:
df["rank_comments_baseline"] = df["relevant_comment_count_x"].rank(
    ascending=False,
    method="dense"
)

df[["video_title","relevant_comment_count_x","rank_comments_baseline"]].sort_values("rank_comments_baseline").head()

Views Ranking (Recommended Baseline)

In [ ]:
df["rank_views_baseline"] = df["views"].rank(
    ascending=False,
    method="dense"
)

df[["video_title","views","rank_views_baseline"]].sort_values("rank_views_baseline").head()

Compare Baselines with Hybrid Model

In [ ]:
cols = [
    "rank_views_baseline",
    "rank_likes_baseline",
    "rank_comments_baseline",
    "final_rank"
]

df[["video_id","video_title"] + cols].sort_values("final_rank").head(10)

Spearman Comparison with Hybrid

In [ ]:
print(df.columns)

In [ ]:
from scipy.stats import spearmanr

baselines = [
    "rank_views_baseline",
    "rank_likes_baseline",
    "rank_comments_baseline"
]

for b in baselines:
    rho, p = spearmanr(df[b], df["final_rank"])
    print(b, "vs Hybrid →", rho)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

for i in range(len(df)):

    plt.plot(
        [1,2],
        [df["rank_views_baseline"].iloc[i], df["final_rank"].iloc[i]],
        marker="o",
        alpha=0.6
    )

plt.xticks([1,2], ["Views Rank", "Hybrid Rank"])
plt.ylabel("Rank Position")
plt.title("Ranking Shift: Views Baseline → Hybrid Model")

plt.gca().invert_yaxis()
plt.grid(True)

plt.show()

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10,6))

plt.scatter(df["rank_views_baseline"], df["final_rank"], label="Views", alpha=0.7)
plt.scatter(df["rank_likes_baseline"], df["final_rank"], label="Likes", alpha=0.7)
plt.scatter(df["rank_comments_baseline"], df["final_rank"], label="Comments", alpha=0.7)

plt.xlabel("Baseline Rank")
plt.ylabel("Hybrid Rank")
plt.title("Baseline Rankings vs Hybrid Ranking")

plt.legend()
plt.grid(True)
plt.gca().invert_yaxis()

plt.show()

Ablation Study Code

In [ ]:
from scipy.stats import spearmanr

print("ABLATION STUDY\n")

rho, p = spearmanr(df["rank_views_baseline"], df["final_rank"])
print("Views vs Hybrid")
print("Spearman:", rho)
print("p-value:", p)
print()

rho, p = spearmanr(df["rank_likes_baseline"], df["final_rank"])
print("Likes vs Hybrid")
print("Spearman:", rho)
print("p-value:", p)
print()

rho, p = spearmanr(df["rank_comments_baseline"], df["final_rank"])
print("Comments vs Hybrid")
print("Spearman:", rho)
print("p-value:", p)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

ablation_matrix = df[
    ["final_engagement_score_x", "ITQS_x", "hybrid_score"]
].corr(method="spearman")

sns.heatmap(ablation_matrix, annot=True, cmap="coolwarm")
plt.title("Ablation Study: Model Component Correlation")
plt.show()

Sensitivity Analysis **Code**

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr

k_values = [1, 2, 3, 5, 10]

results = []

for k in k_values:

    alpha_k = np.log1p(df["relevant_comment_count_x"]) / (
        np.log1p(df["relevant_comment_count_x"]) + k
    )

    hybrid_k = alpha_k * df["final_engagement_score_x"] + \
               (1 - alpha_k) * df["ITQS_x"]

    rho, p = spearmanr(df["hybrid_score"], hybrid_k)

    results.append({
        "k": k,
        "Spearman_with_original": rho,
        "p_value": p
    })

sens_df = pd.DataFrame(results)

sens_df

In [ ]:
import matplotlib.pyplot as plt

plt.plot(sens_df["k"], sens_df["Spearman_with_original"], marker="o")

plt.xlabel("k value")
plt.ylabel("Spearman Correlation with Original Hybrid")
plt.title("Sensitivity Analysis of k")
plt.grid(True)

plt.show()